# Lab 3, Student Performance Dataset 

Team Members: Sandro Juric & Scotty Seethoff

### AI usage disclaimer
We have used the Copilot in this project, primarily for code refactoring in VS Code

# Business Understanding

This is syntheticly generate dataset of 5000 student with several key demographic data including age, gender, academic level.  Accompanied by several features we can use to predict classifies of focus index, burnout level, productivity score, and exam score. Assuming we can build a accurate prediction model that can determine these classes based on features of number of study hours, self-study hours, online classes, number of social media hours, gaming hours, sleep hours, screen time hours, excise amounts, caffeine intake, internet quality, and mental health score once we are able to obtain real time data we could better devise plans to help out students be more successful, productive, and increase their focus index, while keeping the burnout level low. Eventhough this data set was generated synthetically, data has relaistic relationship between features that can be used to simulate real world scenerios.


Dataset: https://www.kaggle.com/datasets/amar5693/student-performance-dataset

Questions we are seeking to answers:

1. Can we predict exam score based on the key features alone?
2. Is the burnout level predictiable based on features above?
3. Can we predict productivity and focus score and are they directly correlated?

# Data Understanding

In [20]:
import pandas as pd
import numpy as np

# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None  # default='warn'
# Ensure future behavior for downcasting is explicit
pd.set_option('future.no_silent_downcasting', True)

df = pd.read_csv('ultimate_student_productivity_dataset_5000.csv')




# select relevant columns
dfSelect = df[['student_id', 'age', 'gender', 'academic_level', 'study_hours', 'self_study_hours', 'online_classes_hours', 'social_media_hours', 'gaming_hours', 'sleep_hours', 'screen_time_hours', 'exercise_minutes', 'caffeine_intake_mg', 'internet_quality', 'mental_health_score', 'focus_index', 'burnout_level', 'productivity_score', 'exam_score']]

gender_map = {
    'Other': 0,
    'Male': 1,
    'Female': 2
}

academic_level_map = {
    'High School': 1,
    'Undergraduate': 2,
    'Graduate': 3,
    'PhD': 4
}

internet_quality_map = {
    'Poor': 1,
    'Average': 2,
    'Good': 3,
    'Excellent': 4 
}

dfSelect['gender'] = dfSelect['gender'].map(gender_map).fillna(0)
dfSelect['academic_level'] = dfSelect['academic_level'].map(academic_level_map).fillna(0)
dfSelect['internet_quality'] = dfSelect['internet_quality'].map(internet_quality_map).fillna(0)


dfSelect.info()
# format and display of the first few rows used the pandas Styler for better visualization in Jupyter Notebooks https://pandas.pydata.org/docs/user_guide/style.html
display(dfSelect.head(10).style.background_gradient(axis=None, cmap="YlGnBu"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   student_id            5000 non-null   int64  
 1   age                   5000 non-null   int64  
 2   gender                5000 non-null   int64  
 3   academic_level        5000 non-null   float64
 4   study_hours           5000 non-null   float64
 5   self_study_hours      5000 non-null   float64
 6   online_classes_hours  5000 non-null   float64
 7   social_media_hours    5000 non-null   float64
 8   gaming_hours          5000 non-null   float64
 9   sleep_hours           5000 non-null   float64
 10  screen_time_hours     5000 non-null   float64
 11  exercise_minutes      5000 non-null   int64  
 12  caffeine_intake_mg    5000 non-null   int64  
 13  internet_quality      5000 non-null   int64  
 14  mental_health_score   5000 non-null   int64  
 15  focus_index          

,student_id,age,gender,academic_level,study_hours,self_study_hours,online_classes_hours,social_media_hours,gaming_hours,sleep_hours,screen_time_hours,exercise_minutes,caffeine_intake_mg,internet_quality,mental_health_score,focus_index,burnout_level,productivity_score,exam_score
0,1,18,0,1.000000,7.640000,1.560000,2.200000,3.050000,2.190000,6.520000,6.470000,81,38,3,10,43.050000,31.770000,73.650000,50.160000
1,2,18,0,1.000000,2.210000,2.220000,2.100000,1.650000,2.550000,5.970000,6.050000,111,339,3,3,15.920000,37.000000,13.700000,1.000000
2,3,22,1,1.000000,3.450000,0.000000,0.290000,1.340000,2.080000,8.390000,7.620000,68,266,3,8,27.390000,34.370000,45.150000,18.300000
3,4,17,0,1.000000,5.750000,2.080000,3.010000,2.270000,2.200000,6.310000,11.670000,113,480,1,3,22.310000,77.310000,20.920000,9.370000
4,5,19,0,1.000000,6.830000,1.720000,3.330000,2.650000,0.700000,8.010000,10.020000,121,24,3,8,38.110000,39.530000,59.230000,27.810000
5,6,25,1,2.000000,2.210000,3.500000,1.690000,4.470000,1.560000,8.340000,8.060000,110,288,2,8,30.520000,33.640000,36.200000,18.530000
6,7,22,0,1.000000,4.600000,2.370000,1.270000,3.120000,1.890000,7.610000,11.340000,19,123,1,6,33.260000,62.010000,37.590000,10.350000
7,8,17,1,1.000000,8.770000,3.780000,2.090000,1.760000,3.130000,7.620000,7.270000,135,379,3,1,31.980000,45.980000,45.040000,19.890000
8,9,16,2,2.000000,6.600000,0.840000,1.000000,4.390000,0.850000,8.290000,8.240000,101,308,2,10,30.230000,49.440000,61.830000,30.730000
9,10,17,2,1.000000,3.980000,0.160000,1.290000,3.200000,2.130000,5.430000,7.790000,142,415,2,9,20.260000,60.780000,29.060000,8.990000
